# Chapter 9: Working with Dates and Times

Companion notebook for *Master Data Cleaning: A Hands-On Beginner's Guide with Python, pandas & NumPy*.

Run the cells from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The data files this chapter uses are in the same folder as this notebook.

## 9.1 Why Dates Need Their Own Type

In [1]:
import pandas as pd
import numpy as np

orders = pd.read_csv('orders.csv')
orders.dtypes

order_id        int64
customer          str
order_date        str
ship_date         str
amount        float64
dtype: object

The next cell raises an error on purpose.

In [2]:
orders['ship_date'] - orders['order_date']

TypeError: operation 'sub' not supported for dtype 'str' with dtype 'str'

### Timestamps, `datetime64` and `NaT`

In [3]:
pd.Timestamp('2024-01-05')

Timestamp('2024-01-05 00:00:00')

In [4]:
pd.Timestamp('2024-03-15') - pd.Timestamp('2024-02-13')

Timedelta('31 days 00:00:00')

## 9.2 Converting Text to Dates with `pd.to_datetime()`

### `pd.to_datetime()`

In [5]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['ship_date'] = pd.to_datetime(orders['ship_date'])
orders.dtypes

order_id               int64
customer                 str
order_date    datetime64[us]
ship_date     datetime64[us]
amount               float64
dtype: object

In [6]:
orders

,order_id,customer,order_date,ship_date,amount
0,1001,Aina,2024-01-05,2024-01-08,250.50
1,1002,Ben,2024-01-18,2024-01-22,175.00
2,1003,Chen,2024-02-02,NaT,420.75
3,1004,Devi,2024-02-14,2024-02-16,310.25
4,1005,Ethan,2024-02-27,2024-03-04,98.40
5,1006,Farah,2024-03-09,2024-03-11,512.00
6,1007,Gopal,2024-03-21,2024-03-25,143.60


### Parsing dates while reading a file

In [7]:
orders = pd.read_csv('orders.csv',
                     parse_dates=['order_date', 'ship_date'])
orders.dtypes

order_id               int64
customer                 str
order_date    datetime64[us]
ship_date     datetime64[us]
amount               float64
dtype: object

### Missing values and empty strings

In [8]:
payment_dates = pd.Series(['2024-05-10', None, '2024-06-15', ''])
pd.to_datetime(payment_dates)

0   2024-05-10
1          NaT
2   2024-06-15
3          NaT
dtype: datetime64[us]

### Telling pandas the format

In [9]:
pd.to_datetime(pd.Series(['03/06/2024']))

0   2024-03-06
dtype: datetime64[us]

In [10]:
hire_dates = pd.Series(['15-03-2024', '22-06-2023', '10-11-2022'])
pd.to_datetime(hire_dates, format='%d-%m-%Y')

0   2024-03-15
1   2023-06-22
2   2022-11-10
dtype: datetime64[us]

In [11]:
event_dates = pd.Series(['15-Jan-2024', '22-Feb-2024', '10-Mar-2024'])
pd.to_datetime(event_dates, format='%d-%b-%Y')

0   2024-01-15
1   2024-02-22
2   2024-03-10
dtype: datetime64[us]

### Handling values that aren't dates

In [12]:
record_dates = pd.Series(['2024-01-15', '2024-13-45',
                          '2024-02-20', 'unknown'])
pd.to_datetime(record_dates, errors='coerce')

0   2024-01-15
1          NaT
2   2024-02-20
3          NaT
dtype: datetime64[us]

### Columns with mixed formats

In [13]:
signup_dates = pd.Series(['2024-03-08', '15/01/2024',
                          'Jan 15, 2024', '08/03/2024'])
pd.to_datetime(signup_dates, errors='coerce')

0   2024-03-08
1          NaT
2          NaT
3          NaT
dtype: datetime64[us]

In [14]:
pd.to_datetime(signup_dates, format='mixed', dayfirst=True)

0   2024-08-03
1   2024-01-15
2   2024-01-15
3   2024-03-08
dtype: datetime64[us]

In [15]:
iso = pd.to_datetime(signup_dates, format='%Y-%m-%d',
                     errors='coerce')
slashed = pd.to_datetime(signup_dates, format='%d/%m/%Y',
                         errors='coerce')
written = pd.to_datetime(signup_dates, format='%b %d, %Y',
                         errors='coerce')
signups = iso.fillna(slashed).fillna(written)
signups

0   2024-03-08
1   2024-01-15
2   2024-01-15
3   2024-03-08
dtype: datetime64[us]

In [16]:
signups.isna().sum()

np.int64(0)

### Building dates from separate columns

In [17]:
customers = pd.DataFrame({
    'customer_id': [701, 702, 703],
    'birth_year': [1990, 1985, 1992],
    'birth_month': [3, 7, 11],
    'birth_day': [15, 22, 8],
})

parts = customers[['birth_year', 'birth_month', 'birth_day']]
parts = parts.rename(columns={'birth_year': 'year',
                              'birth_month': 'month',
                              'birth_day': 'day'})
customers['birth_date'] = pd.to_datetime(parts)
customers

,customer_id,birth_year,birth_month,birth_day,birth_date
0,701,1990,3,15,1990-03-15
1,702,1985,7,22,1985-07-22
2,703,1992,11,8,1992-11-08


In [18]:
staff = pd.DataFrame({
    'employee_id': [801, 802, 803],
    'join_year': [2020, 2021, 2019],
    'join_month': [3, 7, 11],
})

staff['join_date'] = pd.to_datetime({
    'year': staff['join_year'],
    'month': staff['join_month'],
    'day': 1,
})
staff

,employee_id,join_year,join_month,join_date
0,801,2020,3,2020-03-01
1,802,2021,7,2021-07-01
2,803,2019,11,2019-11-01


## 9.3 Extracting Parts of a Date with the `.dt` Accessor

### Year, month, day and quarter

In [19]:
order_parts = orders[['order_id', 'order_date']].copy()
order_parts['year'] = orders['order_date'].dt.year
order_parts['month'] = orders['order_date'].dt.month
order_parts['day'] = orders['order_date'].dt.day
order_parts['month_name'] = orders['order_date'].dt.month_name()
order_parts['quarter'] = orders['order_date'].dt.quarter
order_parts

,order_id,order_date,year,month,day,month_name,quarter
0,1001,2024-01-05,2024,1,5,January,1
1,1002,2024-01-18,2024,1,18,January,1
2,1003,2024-02-02,2024,2,2,February,1
3,1004,2024-02-14,2024,2,14,February,1
4,1005,2024-02-27,2024,2,27,February,1
5,1006,2024-03-09,2024,3,9,March,1
6,1007,2024-03-21,2024,3,21,March,1


### Day of the week

In [20]:
week = pd.Series(pd.date_range('2024-06-03', periods=7))
pd.DataFrame({'date': week,
              'dayofweek': week.dt.dayofweek,
              'day_name': week.dt.day_name()})

,date,dayofweek,day_name
0,2024-06-03,0,Monday
1,2024-06-04,1,Tuesday
2,2024-06-05,2,Wednesday
3,2024-06-06,3,Thursday
4,2024-06-07,4,Friday
5,2024-06-08,5,Saturday
6,2024-06-09,6,Sunday


In [21]:
orders[orders['order_date'].dt.dayofweek >= 5]

,order_id,customer,order_date,ship_date,amount
5,1006,Farah,2024-03-09,2024-03-11,512.0


### Hours, minutes and seconds

In [22]:
logins = pd.Series(pd.to_datetime([
    '2024-05-15 09:30:45', '2024-05-15 14:22:15',
    '2024-05-16 08:15:30', '2024-05-16 18:45:00',
]))
pd.DataFrame({'timestamp': logins,
              'hour': logins.dt.hour,
              'minute': logins.dt.minute,
              'second': logins.dt.second})

,timestamp,hour,minute,second
0,2024-05-15 09:30:45,9,30,45
1,2024-05-15 14:22:15,14,22,15
2,2024-05-16 08:15:30,8,15,30
3,2024-05-16 18:45:00,18,45,0


### Yes/no questions about dates

In [23]:
check_dates = pd.Series(pd.to_datetime([
    '2024-02-29', '2024-03-15', '2023-06-30', '2024-01-01',
]))
pd.DataFrame({'date': check_dates,
              'leap_year': check_dates.dt.is_leap_year,
              'month_end': check_dates.dt.is_month_end,
              'month_start': check_dates.dt.is_month_start})

,date,leap_year,month_end,month_start
0,2024-02-29,True,True,False
1,2024-03-15,True,False,False
2,2023-06-30,False,True,False
3,2024-01-01,True,False,True


### The most common `.dt` error

The next cell raises an error on purpose.

In [24]:
raw_dates = pd.Series(['2024-06-03', '2024-06-04'])
raw_dates.dt.year

AttributeError: Can only use .dt accessor with datetimelike values

## 9.4 Handling Missing Dates and Formatting Dates

### `s.isna()`

In [25]:
orders[['order_date', 'ship_date']].isna().sum()

order_date    0
ship_date     1
dtype: int64

In [26]:
orders[orders['ship_date'].isna()]

,order_id,customer,order_date,ship_date,amount
2,1003,Chen,2024-02-02,NaT,420.75


### `df.dropna(subset=...)`

In [27]:
shipped = orders.dropna(subset=['ship_date'])
shipped

,order_id,customer,order_date,ship_date,amount
0,1001,Aina,2024-01-05,2024-01-08,250.50
1,1002,Ben,2024-01-18,2024-01-22,175.00
3,1004,Devi,2024-02-14,2024-02-16,310.25
4,1005,Ethan,2024-02-27,2024-03-04,98.40
5,1006,Farah,2024-03-09,2024-03-11,512.00
6,1007,Gopal,2024-03-21,2024-03-25,143.60


### `s.fillna()`

In [28]:
accounts = pd.DataFrame({
    'account_id': [3001, 3002, 3003, 3004],
    'created': pd.to_datetime(['2024-01-15', None,
                               '2024-03-10', '']),
})
accounts['created'] = accounts['created'].fillna(
    pd.Timestamp('1900-01-01'))
accounts

,account_id,created
0,3001,2024-01-15
1,3002,1900-01-01
2,3003,2024-03-10
3,3004,1900-01-01


### `s.ffill()` and `s.bfill()`

In [29]:
status_log = pd.DataFrame({
    'record_id': [4001, 4002, 4003, 4004, 4005, 4006],
    'status_date': pd.to_datetime(['2024-01-15', None, None,
                                   '2024-02-20', None,
                                   '2024-03-10']),
})
status_log['ffill'] = status_log['status_date'].ffill()
status_log['ffill_limit_1'] = status_log['status_date'].ffill(
    limit=1)
status_log['bfill'] = status_log['status_date'].bfill()
status_log

,record_id,status_date,ffill,ffill_limit_1,bfill
0,4001,2024-01-15,2024-01-15,2024-01-15,2024-01-15
1,4002,NaT,2024-01-15,2024-01-15,2024-02-20
2,4003,NaT,2024-01-15,NaT,2024-02-20
3,4004,2024-02-20,2024-02-20,2024-02-20,2024-02-20
4,4005,NaT,2024-02-20,2024-02-20,2024-03-10
5,4006,2024-03-10,2024-03-10,2024-03-10,2024-03-10


### `s.dt.strftime()`

In [30]:
orders_report = orders[['order_id', 'order_date']].copy()
orders_report['long'] = orders['order_date'].dt.strftime(
    '%B %d, %Y')
orders_report['us_style'] = orders['order_date'].dt.strftime(
    '%m/%d/%Y')
orders_report.head(3)

,order_id,order_date,long,us_style
0,1001,2024-01-05,"January 05, 2024",01/05/2024
1,1002,2024-01-18,"January 18, 2024",01/18/2024
2,1003,2024-02-02,"February 02, 2024",02/02/2024


### Putting it together

In [31]:
shipments = pd.DataFrame({
    'order_id': [8001, 8002, 8003, 8004, 8005],
    'order_date': ['2024-01-15', '', None, '2024-02-10', None],
    'ship_date': ['2024-01-18', '2024-01-22', '', None,
                  '2024-02-25'],
})
shipments['order_date'] = pd.to_datetime(shipments['order_date'])
shipments['ship_date'] = pd.to_datetime(shipments['ship_date'])

shipments['order_date'] = shipments['order_date'].ffill(limit=1)
shipments['ship_date'] = shipments['ship_date'].fillna(
    shipments['order_date'] + pd.Timedelta(days=3))
shipments['order_display'] = shipments['order_date'].dt.strftime(
    '%d %b %Y')
shipments

,order_id,order_date,ship_date,order_display
0,8001,2024-01-15,2024-01-18,15 Jan 2024
1,8002,2024-01-15,2024-01-22,15 Jan 2024
2,8003,NaT,NaT,NaN
3,8004,2024-02-10,2024-02-13,10 Feb 2024
4,8005,2024-02-10,2024-02-25,10 Feb 2024


## 9.5 Date Arithmetic and Time Differences

### Subtracting dates

In [32]:
delivery = orders[['order_id', 'order_date', 'ship_date']].copy()
delivery['days_to_ship'] = (
    delivery['ship_date'] - delivery['order_date']).dt.days
delivery

,order_id,order_date,ship_date,days_to_ship
0,1001,2024-01-05,2024-01-08,3.0
1,1002,2024-01-18,2024-01-22,4.0
2,1003,2024-02-02,NaT,NaN
3,1004,2024-02-14,2024-02-16,2.0
4,1005,2024-02-27,2024-03-04,6.0
5,1006,2024-03-09,2024-03-11,2.0
6,1007,2024-03-21,2024-03-25,4.0


In [33]:
delivery['days_to_ship'].mean()

np.float64(3.5)

### `s.dt.total_seconds()`

In [34]:
shifts = pd.DataFrame({
    'shift_id': [1, 2, 3],
    'start': pd.to_datetime(['2024-05-15 09:00',
                             '2024-05-15 10:30',
                             '2024-05-15 14:15']),
    'end': pd.to_datetime(['2024-05-15 11:30',
                           '2024-05-15 12:45',
                           '2024-05-15 18:20']),
})
duration = shifts['end'] - shifts['start']
shifts['hours'] = duration.dt.total_seconds() / 3600
shifts['minutes'] = duration.dt.total_seconds() / 60
shifts

,shift_id,start,end,hours,minutes
0,1,2024-05-15 09:00:00,2024-05-15 11:30:00,2.500000,150.0
1,2,2024-05-15 10:30:00,2024-05-15 12:45:00,2.250000,135.0
2,3,2024-05-15 14:15:00,2024-05-15 18:20:00,4.083333,245.0


In [35]:
pd.Timedelta(days=1, hours=1).total_seconds()

90000.0

### `pd.Timedelta()` and `pd.to_timedelta()`

In [36]:
contracts = pd.DataFrame({
    'contract_id': [2001, 2002, 2003],
    'start': pd.to_datetime(['2024-01-15', '2024-02-01',
                             '2024-03-10']),
    'length_days': [30, 60, 90],
})
contracts['end_30'] = contracts['start'] + pd.Timedelta(days=30)
contracts['end'] = contracts['start'] + pd.to_timedelta(
    contracts['length_days'], unit='D')
contracts

,contract_id,start,length_days,end_30,end
0,2001,2024-01-15,30,2024-02-14,2024-02-14
1,2002,2024-02-01,60,2024-03-02,2024-04-01
2,2003,2024-03-10,90,2024-04-09,2024-06-08


### `pd.DateOffset()`

In [37]:
payments = pd.DataFrame({
    'first_payment': pd.to_datetime(['2024-01-31', '2024-02-29',
                                     '2024-03-31']),
})
payments['second_payment'] = (
    payments['first_payment'] + pd.DateOffset(months=1))
payments

,first_payment,second_payment
0,2024-01-31,2024-02-29
1,2024-02-29,2024-03-29
2,2024-03-31,2024-04-30


### `pd.offsets.MonthEnd()`

In [38]:
some_dates = pd.Series(pd.to_datetime(['2024-01-15', '2024-01-31',
                                       '2024-02-10']))
pd.DataFrame({'date': some_dates,
              'month_end_0': some_dates + pd.offsets.MonthEnd(0),
              'month_end_1': some_dates + pd.offsets.MonthEnd(1)})

,date,month_end_0,month_end_1
0,2024-01-15,2024-01-31,2024-01-31
1,2024-01-31,2024-01-31,2024-02-29
2,2024-02-10,2024-02-29,2024-02-29


In [39]:
month_end = orders['order_date'] + pd.offsets.MonthEnd(0)
orders.groupby(month_end)['amount'].sum()

order_date
2024-01-31    425.5
2024-02-29    829.4
2024-03-31    655.6
Name: amount, dtype: float64

### `pd.date_range()`

In [40]:
pd.date_range(start='2024-06-01', end='2024-06-05', freq='D')

DatetimeIndex(['2024-06-01', '2024-06-02', '2024-06-03', '2024-06-04',
               '2024-06-05'],
              dtype='datetime64[us]', freq='D')

In [41]:
pd.date_range(start='2024-06-01', periods=5, freq='W')

DatetimeIndex(['2024-06-02', '2024-06-09', '2024-06-16', '2024-06-23',
               '2024-06-30'],
              dtype='datetime64[us]', freq='W-SUN')

In [42]:
calendar = pd.DataFrame(
    {'date': pd.date_range('2024-07-01', periods=7, freq='D')})
calendar['day_name'] = calendar['date'].dt.day_name()
calendar['is_weekend'] = calendar['date'].dt.dayofweek >= 5
calendar

,date,day_name,is_weekend
0,2024-07-01,Monday,False
1,2024-07-02,Tuesday,False
2,2024-07-03,Wednesday,False
3,2024-07-04,Thursday,False
4,2024-07-05,Friday,False
5,2024-07-06,Saturday,True
6,2024-07-07,Sunday,True


## 9.6 Shifting and Rolling Calculations

In [43]:
revenue = pd.DataFrame({
    'revenue': [1000, 1200, 1150, 1300, 1250,
                1400, 1350, 1100, 1300, 1250],
}, index=pd.date_range('2024-01-01', periods=10, freq='D'))
revenue.index.name = 'date'
revenue

,revenue
date,
2024-01-01,1000
2024-01-02,1200
2024-01-03,1150
2024-01-04,1300
2024-01-05,1250
2024-01-06,1400
2024-01-07,1350
2024-01-08,1100
2024-01-09,1300


### `s.shift()`

In [44]:
revenue['prev_day'] = revenue['revenue'].shift(1)
revenue['change'] = revenue['revenue'] - revenue['prev_day']
revenue['growth_pct'] = (
    revenue['change'] / revenue['prev_day'] * 100).round(2)
revenue.head()

,revenue,prev_day,change,growth_pct
date,,,,
2024-01-01,1000,NaN,NaN,NaN
2024-01-02,1200,1000.0,200.0,20.00
2024-01-03,1150,1200.0,-50.0,-4.17
2024-01-04,1300,1150.0,150.0,13.04
2024-01-05,1250,1300.0,-50.0,-3.85


In [45]:
revenue['next_day'] = revenue['revenue'].shift(-1)
revenue[['revenue', 'next_day']].tail(3)

,revenue,next_day
date,,
2024-01-08,1100,1300.0
2024-01-09,1300,1250.0
2024-01-10,1250,NaN


In [46]:
revenue['last_week'] = revenue['revenue'].shift(7)
revenue['week_change'] = revenue['revenue'] - revenue['last_week']
revenue[['revenue', 'last_week', 'week_change']].tail()

,revenue,last_week,week_change
date,,,
2024-01-06,1400,NaN,NaN
2024-01-07,1350,NaN,NaN
2024-01-08,1100,1000.0,100.0
2024-01-09,1300,1200.0,100.0
2024-01-10,1250,1150.0,100.0


### `s.rolling()`

In [47]:
revenue['total_3d'] = revenue['revenue'].rolling(window=3).sum()
revenue[['revenue', 'total_3d']].head()

,revenue,total_3d
date,,
2024-01-01,1000,NaN
2024-01-02,1200,NaN
2024-01-03,1150,3350.0
2024-01-04,1300,3650.0
2024-01-05,1250,3700.0


In [48]:
revenue['avg_3d'] = (
    revenue['revenue'].rolling(window=3).mean().round(2))
revenue['avg_7d'] = (
    revenue['revenue'].rolling(window=7).mean().round(2))
revenue[['revenue', 'avg_3d', 'avg_7d']].tail()

,revenue,avg_3d,avg_7d
date,,,
2024-01-06,1400,1316.67,NaN
2024-01-07,1350,1333.33,1235.71
2024-01-08,1100,1283.33,1250.00
2024-01-09,1300,1250.00,1264.29
2024-01-10,1250,1216.67,1278.57


In [49]:
revenue['avg_3d_partial'] = revenue['revenue'].rolling(
    window=3, min_periods=1).mean().round(2)
revenue[['revenue', 'avg_3d_partial']].head()

,revenue,avg_3d_partial
date,,
2024-01-01,1000,1000.00
2024-01-02,1200,1100.00
2024-01-03,1150,1116.67
2024-01-04,1300,1216.67
2024-01-05,1250,1233.33


### Combining shift and rolling

In [50]:
revenue['avg_3d_change'] = (
    revenue['avg_3d'] - revenue['avg_3d'].shift(1)).round(2)
revenue[['revenue', 'avg_3d', 'avg_3d_change']].tail()

,revenue,avg_3d,avg_3d_change
date,,,
2024-01-06,1400,1316.67,83.34
2024-01-07,1350,1333.33,16.66
2024-01-08,1100,1283.33,-50.00
2024-01-09,1300,1250.00,-33.33
2024-01-10,1250,1216.67,-33.33


## 9.7 Periods, Frequencies and Resampling

### Frequency codes

In [51]:
pd.date_range('2024-01-05', periods=5, freq='B')

DatetimeIndex(['2024-01-05', '2024-01-08', '2024-01-09', '2024-01-10',
               '2024-01-11'],
              dtype='datetime64[us]', freq='B')

In [52]:
pd.date_range('2024-01-01', periods=4, freq='4h')

DatetimeIndex(['2024-01-01 00:00:00', '2024-01-01 04:00:00',
               '2024-01-01 08:00:00', '2024-01-01 12:00:00'],
              dtype='datetime64[us]', freq='4h')

### `pd.Period()`

In [53]:
march = pd.Period('2024-03', freq='M')
print(march)
print(march.start_time)
print(march.end_time)

2024-03
2024-03-01 00:00:00
2024-03-31 23:59:59.999999


In [54]:
print(march + 1)
print(pd.Period('2024Q2', freq='Q') - 1)
print(pd.Period('2024-06', freq='M') - march)

2024-04
2024Q1
<3 * MonthEnds>


### `pd.period_range()`

In [55]:
pd.period_range(start='2024-01', end='2024-04', freq='M')

PeriodIndex(['2024-01', '2024-02', '2024-03', '2024-04'], dtype='period[M]')

### `df.to_period()` and `df.to_timestamp()`

In [56]:
monthly_sales = pd.DataFrame({
    'sales': [45000, 52000, 48000, 61000, 57000, 63000],
}, index=pd.date_range('2024-01-31', periods=6, freq='ME'))
monthly_sales

,sales
2024-01-31,45000
2024-02-29,52000
2024-03-31,48000
2024-04-30,61000
2024-05-31,57000
2024-06-30,63000


In [57]:
by_period = monthly_sales.to_period('M')
by_period

,sales
2024-01,45000
2024-02,52000
2024-03,48000
2024-04,61000
2024-05,57000
2024-06,63000


In [58]:
by_period.to_timestamp().head(3)

,sales
2024-01-01,45000
2024-02-01,52000
2024-03-01,48000


In [59]:
by_period.to_timestamp(how='end').head(3)

,sales
2024-01-31 23:59:59.999999,45000
2024-02-29 23:59:59.999999,52000
2024-03-31 23:59:59.999999,48000


### `df.resample()`

In [60]:
rng = np.random.default_rng(42)
daily_sales = pd.DataFrame({
    'revenue': rng.integers(1000, 1500, size=91),
    'units': rng.integers(50, 80, size=91),
}, index=pd.date_range('2024-01-01', periods=91, freq='D'))
daily_sales.index.name = 'date'
daily_sales.head()

,revenue,units
date,,
2024-01-01,1044,74
2024-01-02,1386,75
2024-01-03,1327,61
2024-01-04,1219,76
2024-01-05,1216,58


In [61]:
daily_sales.resample('W').sum().head(3)

,revenue,units
date,,
2024-01-07,8663,471
2024-01-14,8992,449
2024-01-21,8965,481


In [62]:
daily_sales.resample('ME').sum()

,revenue,units
date,,
2024-01-31,39400,2010
2024-02-29,36327,1874
2024-03-31,39103,1995


In [63]:
daily_sales.resample('ME').mean().round(2)

,revenue,units
date,,
2024-01-31,1270.97,64.84
2024-02-29,1252.66,64.62
2024-03-31,1261.39,64.35


In [64]:
daily_sales.resample('ME').agg({'revenue': 'sum', 'units': 'mean'})

,revenue,units
date,,
2024-01-31,39400,64.838710
2024-02-29,36327,64.620690
2024-03-31,39103,64.354839


### Weeks that start on Monday

In [65]:
weekly_mon = daily_sales.resample('W-MON', closed='left',
                                  label='left').sum()
weekly_mon.head(3)

,revenue,units
date,,
2024-01-01,8663,471
2024-01-08,8992,449
2024-01-15,8965,481


### Upsampling

In [66]:
daily_from_monthly = monthly_sales.resample('D')
daily_from_monthly.asfreq().head()

,sales
2024-01-31,45000.0
2024-02-01,NaN
2024-02-02,NaN
2024-02-03,NaN
2024-02-04,NaN


In [67]:
daily_from_monthly.ffill().head()

,sales
2024-01-31,45000
2024-02-01,45000
2024-02-02,45000
2024-02-03,45000
2024-02-04,45000


In [68]:
daily_from_monthly.interpolate().iloc[27:32].round(2)

,sales
2024-02-27,51517.24
2024-02-28,51758.62
2024-02-29,52000.00
2024-03-01,51870.97
2024-03-02,51741.94


## 9.8 Time Zones

### Naive and aware timestamps

In [69]:
readings = pd.DataFrame({
    'temp_c': [22.1, 23.4, 24.8, 25.3, 24.9],
}, index=pd.date_range('2024-03-15 09:00', periods=5, freq='h'))
readings.index.name = 'time'
print(readings.index.tz)

None


### `df.tz_localize()`

In [70]:
readings_utc = readings.tz_localize('UTC')
readings_utc.head(3)

,temp_c
time,
2024-03-15 09:00:00+00:00,22.1
2024-03-15 10:00:00+00:00,23.4
2024-03-15 11:00:00+00:00,24.8


In [71]:
readings_ny = readings.tz_localize('America/New_York')
readings_ny.head(3)

,temp_c
time,
2024-03-15 09:00:00-04:00,22.1
2024-03-15 10:00:00-04:00,23.4
2024-03-15 11:00:00-04:00,24.8


### `df.tz_convert()`

In [72]:
readings_ny.tz_convert('UTC').head(3)

,temp_c
time,
2024-03-15 13:00:00+00:00,22.1
2024-03-15 14:00:00+00:00,23.4
2024-03-15 15:00:00+00:00,24.8


### Combining data from different time zones

In [73]:
ny_sales = pd.DataFrame({
    'sales': [1200, 1800],
    'store': 'New York',
}, index=pd.to_datetime(['2024-03-15 09:00', '2024-03-15 11:00']))

london_sales = pd.DataFrame({
    'sales': [900, 950],
    'store': 'London',
}, index=pd.to_datetime(['2024-03-15 14:00', '2024-03-15 16:00']))

In [74]:
ny_utc = ny_sales.tz_localize('America/New_York').tz_convert('UTC')
london_utc = (london_sales.tz_localize('Europe/London')
              .tz_convert('UTC'))
all_sales = pd.concat([ny_utc, london_utc]).sort_index()
all_sales

,sales,store
2024-03-15 13:00:00+00:00,1200,New York
2024-03-15 14:00:00+00:00,900,London
2024-03-15 15:00:00+00:00,1800,New York
2024-03-15 16:00:00+00:00,950,London


### Mixing naive and aware timestamps

The next cell raises an error on purpose.

In [75]:
naive = pd.Timestamp('2024-03-15 09:00')
aware = pd.Timestamp('2024-03-15 09:00', tz='America/New_York')
aware - naive

TypeError: Cannot subtract tz-naive and tz-aware datetime-like objects.